# Data Science – Exercise Session 2

Plan for today:

1. **Advent of Code**: a fun way to practice programming (and parsing!)
2. **Web scraping, continued**: we finish the Pokémon table from last week's notebook
3. **NumPy & pandas**
4. **E2**

Run the cell below first to make sure all the libraries are installed.

In [ ]:
import numpy as np
import pandas as pd
import bs4

print("numpy", np.__version__, "| pandas", pd.__version__, "| beautifulsoup4", bs4.__version__)

---
# Part 1: Practicing with Advent of Code

[Advent of Code](https://adventofcode.com) (AoC) is an advent calendar of programming puzzles, run by Eric Wastl since 2015.

* Each day unlocks a puzzle with **two parts**. You only see part 2 after you've solved part 1, and it usually adds a twist to part 1.
* Every participant gets their **own puzzle input**, a text file, and you submit only the answer (a number or a string). You can solve it in any language.
* Since 2025 there are **12 puzzles per year** (1–12 December). There is no global leaderboard any more, only private leaderboards among friends or classmates.
* The **archive is always open**: every puzzle since 2015 can be solved right now. The 2026 edition starts on 1 December.

**Why this is good practice for this course**

* Every puzzle starts with *parsing a text file into a data structure*, which tests web scraping skills.
* You have to choose suitable data structures (lists, dicts, sets, `Counter`, ...) and later algorithms.
* The puzzles get harder over the month, and the first few days of each year are suitable for beginners.

**Ground rules**

* Don't publish your puzzle input; the author asks people not to redistribute them.
* Solve it yourself. Pasting the puzzle into an LLM gets you a star but not the skill.
* Be polite to the server: download your input once and save it to a file.

### A good workflow

1. Read the puzzle carefully. The **small example in the puzzle text is your test case**.
2. Write a `parse(text)` function that turns the raw text into Python data.
3. Write `part1(...)`, and check it w on the example **before** running it on your real input.
4. Submit, unlock part 2, and repeat step 3.

In [ ]:
example = '''
'''

def parse(text):
    return

parse(example)

In [1]:
from collections import Counter

def part1(txtpath):
    # sorted() pairs the smallest with the smallest, etc.; zip() walks both lists together
    return

with open("input.txt", "r") as txtfile:
    filestring = txtfile.read()
    data = [[int(d) for d in s.split("\n")] for s in filestring.split("\n\n")]

elfcalorie = [sum(elf) for elf in data]
print("The answer is ", max(elfcalorie))

FileNotFoundError: [Errno 2] No such file or directory: 'input.txt'

Once the example passes, we run it on the real input. Log in, open <https://adventofcode.com/2024/day/1/input>, and save the page as `input-2024-01.txt` next to this notebook.

In [ ]:
from pathlib import Path

input_file = Path("input-2024-01.txt")
if input_file.exists():
    left, right = parse(input_file.read_text())
    print("Part 1:", part1(left, right))
    print("Part 2:", part2(left, right))
else:
    print(f"Save your puzzle input as {input_file} to get your answers.")

**Preview of Part 3:** the same puzzle in NumPy. `np.loadtxt` parses the columns for us, and *broadcasting* replaces the counting loop. Don't worry if this isn't clear yet; we'll come back to it.

In [ ]:
import io

grid = np.loadtxt(io.StringIO(example), dtype=int)   # shape (6, 2)
left, right = grid.T                                  # transpose -> two rows -> unpack into two arrays

print("part 1:", np.abs(np.sort(left) - np.sort(right)).sum())
# (6, 1) == (1, 6) broadcasts to a (6, 6) boolean matrix; summing each row counts the matches
print("part 2:", (left * (left[:, None] == right[None, :]).sum(axis=1)).sum())

### Your turn: 2015, Day 1 ("Not Quite Lisp")

Read the puzzle at <https://adventofcode.com/2015/day/1>. In short: Santa starts on floor 0 of a building. Each `(` means "go up one floor", and each `)` means "go down one floor".

* **Part 1:** on which floor does Santa end up?
* **Part 2:** at which *position* (the first character has position 1) does Santa enter the basement (floor -1) for the first time?

Fill in the two functions. The `check` helper runs them on the examples from the puzzle text.

In [ ]:
def check(fn, cases):
    # print a ✓/✗ line per example instead of stopping at the first failure
    for arg, expected in cases.items():
        got = fn(arg)
        print(f"{'✓' if got == expected else '✗'} {fn.__name__}({arg!r}) = {got!r}, expected {expected!r}")

def final_floor(directions):
    # your code goes here
    pass

def first_basement_position(directions):
    # your code goes here
    pass

check(final_floor, {"(())": 0, "()()": 0, "(((": 3, "(()(()(": 3, "))(((((": 3, "())": -1, ")))": -3, ")())())": -3})
check(first_basement_position, {")": 1, "()())": 5})

<details>
<summary><b>Solution</b> (try it yourself first!)</summary>

```python
def final_floor(directions):
    return directions.count("(") - directions.count(")")

def first_basement_position(directions):
    floor = 0
    for position, char in enumerate(directions, start=1):
        floor += 1 if char == "(" else -1
        if floor == -1:
            return position
```
</details>

**Where to go from here:** work through the first ~5 days of any year, e.g. 2015 or 2024. When you're stuck (or after you've solved a puzzle), [r/adventofcode](https://www.reddit.com/r/adventofcode/) has a solution thread for every day. Comparing your solution with others' is a great way to learn new Python idioms (`collections`, `itertools`, `re`, ...).

---
# Part 2: Web scraping, continued – the Pokémon table

Last week we stopped at this task: turn the table on [List of generation I Pokémon](https://en.wikipedia.org/wiki/List_of_generation_I_Pok%C3%A9mon) into a **list of dictionaries**, one per Pokémon, keyed by the column names:

```python
[{"name": "Bulbasaur", "Type(s)": "Grass / Poison", "Evolves from": "...", "Evolves into": "Ivysaur ...", "Notes": "..."}, ...]
```

### Step 0: look before you code

Open the page, right-click the table, and choose *Inspect*. Here's what you'll find:

* The table we want is the first `<table class="wikitable">`. **It's not the first `<table>`**: the box in the top right ("List of Pokémon by ...") is also a table.
* Each row starts with a `<th>` holding the name: the English name in `<b>`, then the Japanese name and the Pokédex number.
* Then comes a `<td>` that is only a coloured box. That's why the header cell "Type(s)" has `colspan="2"`.
* Pokémon that don't evolve have a single cell "No evolution" with `colspan="2"` instead of separate "Evolves from" and "Evolves into" cells. **These rows have 5 cells instead of 6.**
* Footnote markers like `[15]` are `<sup class="reference">` elements that we don't want in our text.
* The list also contains the glitch Pokémon *MissingNo.*, so we expect 151 + 1 = 152 rows.

### Step 1: download the page (once)

A small fix to last week's starter code: `con.read()` returns **bytes**, and `str(html)` doesn't convert them to text. It gives you their *representation*, `"b'<!DOCTYPE html>\\n<html ..."`, with literal `\n` escapes. Use `.decode("utf-8")` instead.

We also cache the page: we only download it if we don't have a local copy yet, so re-running the cell doesn't send another request to Wikipedia.

In [ ]:
import urllib.request
from pathlib import Path
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/List_of_generation_I_Pok%C3%A9mon"
cache = Path("pokemon-list.html")

if not cache.exists():
    request = urllib.request.Request(url, headers={'User-Agent': "firefox"})
    with urllib.request.urlopen(request) as con:
        html = con.read().decode("utf-8")   # bytes -> str
    cache.write_text(html, encoding="utf-8")

# from here on everything is a local operation
soup = BeautifulSoup(cache.read_text(encoding="utf-8"), 'html.parser')

### Step 2: find the table and clean it up

In [ ]:
table = soup.find("table", class_="wikitable")

# remove all footnote markers ([15], [nb 2], ...) from the tree
for sup in table.find_all("sup", class_="reference"):
    sup.decompose()

header_row, *body_rows = table.find_all("tr")
print([th.get_text(" ", strip=True) for th in header_row.find_all("th")])
print(len(body_rows), "body rows")

### Step 3: explore a few rows before writing the loop

`recursive=False` only returns the *direct* children of the row, so we don't accidentally pick up cells of nested tables.

In [ ]:
def show_row(row):
    for cell in row.find_all(["th", "td"], recursive=False):
        print(f"  <{cell.name} colspan={cell.get('colspan')}>  {cell.get_text(' ', strip=True)[:60]!r}")

print("Bulbasaur:"); show_row(body_rows[0])
print("Farfetch'd:"); show_row(body_rows[82])
print("Last row:"); show_row(body_rows[-1])

Two things to note:

* The very last `<tr>` is empty, so we skip rows without cells.
* For the name we only want the bold part (`cell.b`).
* The Pokédex number, e.g. `(0001)`, is the last number in the name cell, and we keep it as an extra column. For most rows it's a link, but MissingNo. has no link (try `show_row`, or inspect it in the browser). So instead of looking for the link we use a [regular expression](https://docs.python.org/3/howto/regex.html): `re.findall(r"\d+", text)` returns every run of digits in the text.

### Step 4: the loop

In [ ]:
import re

pokemon = []
for row in body_rows:
    cells = row.find_all(["th", "td"], recursive=False)
    if not cells:                                # skip the empty row at the end
        continue

    name_cell, _colour, type_cell, *rest = cells  # rest = [from, into, notes] or [no-evolution, notes]
    if len(rest) == 2:                           # "No evolution" spans both evolution columns
        evolves_from = evolves_into = rest[0].get_text(" ", strip=True)
    else:
        evolves_from = rest[0].get_text(" ", strip=True)
        evolves_into = rest[1].get_text(" ", strip=True)

    pokemon.append({
        "name": name_cell.b.get_text(strip=True),
        "number": int(re.findall(r"\d+", name_cell.get_text())[-1]),   # last number in the cell
        "Type(s)": type_cell.get_text(" ", strip=True),
        "Evolves from": evolves_from,
        "Evolves into": evolves_into,
        "Notes": rest[-1].get_text(" ", strip=True),
    })

print(len(pokemon), "Pokémon")
pokemon[0]

In [ ]:
# sanity checks: a non-evolving Pokémon and the last entry
pokemon[82], pokemon[-1]["name"]

### Wrap-up

* You'll notice that the evolution columns contain `"— N/a"` where there is nothing to evolve from/into: the dash is visible on the page, and "N/a" is hidden text for screen readers. We'll clean that up with pandas in a minute.
* This code relies on the page's layout **today**. If Wikipedia changes the table, the script breaks, which is the brittleness we discussed last week. Always explore rows first and check your result (here: 152 entries, plus spot checks).
* pandas has a shortcut, `pd.read_html(...)`, that turns every `<table>` into a DataFrame and handles `colspan`/`rowspan`. It needs an extra parser (`pip install lxml`), and you have less control over the cleanup. For E2 you'll write the parsing yourself.

---
# Part 3: NumPy & pandas recap

## NumPy

### Why arrays? Vectorization

A NumPy array holds elements of a single type in contiguous memory. Operations on whole arrays run in compiled C code instead of a Python loop.

(Two notes on the timing cell in the NumPy notebook: `perf_counter()` measures **seconds**, not ms, and with `N = 100_000_000` the Python list alone needs several GB of RAM.)

In [ ]:
N = 1_000_000
values = list(range(N))
arr = np.arange(N)

# sum of squares, once with a Python loop and once vectorized: same result?
sum(x * x for x in values) == (arr * arr).sum()

A single `perf_counter()` measurement is noisy. The `%timeit` *magic* runs a line many times and reports the average:

In [ ]:
%timeit sum(x * x for x in values)

In [ ]:
%timeit (arr * arr).sum()

⚠️ Try changing `N` to `10_000_000`: the comparison becomes `False`! The sum of squares becomes larger than the biggest `int64` ($2^{63} - 1 \approx 9.2 \cdot 10^{18}$), and NumPy **silently overflows**. Python's own `int` has no size limit. Speed comes at a price, so keep an eye on your dtypes.

### Descriptive statistics and the `axis` argument

These are the functions you'll use to **check** your own implementations in E2. Here we use a small made-up matrix: 3 sensors (rows) with 5 measurements each (columns).

* `axis=None` (the default) treats the matrix as one long vector.
* `axis=1` computes "along the columns", i.e. **one result per row**.
* `axis=0` computes **one result per column**.

In [ ]:
readings = np.array([
    [2.0, 4.0, 4.0, 5.0, 10.0],   # sensor A
    [1.0, 1.5, 2.0, 2.5,  3.0],   # sensor B
    [7.0, 7.0, 7.0, 7.0,  7.0],   # sensor C
])

print("shape:            ", readings.shape)
print("mean (all values):", np.mean(readings))
print("mean per row:     ", np.mean(readings, axis=1))
print("median per row:   ", np.median(readings, axis=1))
print("variance per row: ", np.var(readings, axis=1))
print("std per row:      ", np.std(readings, axis=1))
print("flattened:        ", readings.ravel())

### Histograms

`np.histogram(data, bins=b)` returns two arrays: the **counts** (length `b`) and the **bin edges** (length `b + 1`). The bins are equally wide between the minimum and maximum. All bins are half-open `[left, right)`, **except the last one, which also includes the maximum.**

In [ ]:
data = [1, 2, 2, 3, 3, 3, 4, 10]
counts, edges = np.histogram(data, bins=3)
print("counts:", counts)   # 1,2,2,3,3,3 -> [1, 4);  4 -> [4, 7);  10 -> [7, 10] (the max lands in the last bin)
print("edges: ", edges)

### Exercises from the NumPy notebook (solutions)

**1.** Two arrays of 10 random integers between 1 and 100: element-wise sum and difference, mean and standard deviation, and the elements of the first array that are greater than the corresponding element of the second.

In [ ]:
rng = np.random.default_rng(seed=42)         # seed -> the same "random" numbers every run
a = rng.integers(1, 101, size=10)            # upper bound is exclusive
b = rng.integers(1, 101, size=10)

print("a     =", a)
print("b     =", b)
print("a + b =", a + b)
print("a - b =", a - b)
print(f"a: mean {a.mean():.1f}, std {a.std():.1f} | b: mean {b.mean():.1f}, std {b.std():.1f}")
print("a[a > b] =", a[a > b])               # boolean mask, compared element by element

**2.** A 5×5 array with the integers 1–25: extract the middle 3×3 block, sum each row and column, and set the diagonal to 0.

In [ ]:
M = np.arange(1, 26).reshape(5, 5)
print(M)
print("middle 3x3:\n", M[1:4, 1:4])
print("row sums:   ", M.sum(axis=1))
print("column sums:", M.sum(axis=0))

M_zero_diag = M.copy()                 # fill_diagonal works in place, so we keep M unchanged
np.fill_diagonal(M_zero_diag, 0)
print(M_zero_diag)

## pandas

Last week we only looked at `Series`. A **DataFrame** is a table of Series that share an index, one Series per column. Our list of dictionaries from Part 2 converts directly into one.

In [ ]:
df = pd.DataFrame(pokemon)
print(df.shape)
df.head()

**Selecting:** `df["col"]` gives a column (a Series). `df.loc[...]` selects rows by **label**, and `df.iloc[...]` selects rows by **position**.

In [ ]:
df = df.set_index("name")               # use the names as row labels
df.loc["Pikachu", ["number", "Type(s)", "Evolves into"]]

In [ ]:
df.iloc[24:27, :3]                      # rows 24-26, first three columns

**Cleaning and new columns:** the `.str` accessor applies string methods to every element of a column.

In [ ]:
df = df.replace("— N/a", "—")                       # replaces exact matches in every column

df["types"] = df["Type(s)"].str.split(" / ")       # "Grass / Poison" -> ["Grass", "Poison"]
df["primary_type"] = df["types"].str[0]
df["dual_type"] = df["types"].str.len() == 2

df[["Type(s)", "types", "primary_type", "dual_type", "Evolves from"]].head()

**Filtering with boolean masks:** this works just like with Series. Combine conditions with `&` (and) and `|` (or), and put each condition in parentheses.

In [ ]:
df[(df["primary_type"] == "Water") & df["dual_type"]][["Type(s)", "Evolves into"]]

**Counting and grouping:**

In [ ]:
# every type, counting dual-type Pokémon once per type: explode() gives each list element its own row
df["types"].explode().value_counts().head(8)

In [ ]:
# per primary type: how many Pokémon, and which fraction of them have two types
(df.groupby("primary_type")["dual_type"]
   .agg(count="size", share_dual="mean")
   .sort_values("count", ascending=False)
   .head(8))

### ⚠️ Careful: NumPy and pandas compute the variance differently by default

* NumPy: `np.var(x)` divides by $N$ (`ddof=0`), which is the formula in E2.
* pandas: `series.var()` divides by $N-1$ (`ddof=1`, the "sample variance").

So **check your E2 results with NumPy**, or pass `ddof=0` to pandas.

In [ ]:
x = pd.Series([1, 2, 3, 4])
print("np.var(x)        =", np.var(x))
print("x.var()          =", x.var())
print("x.var(ddof=0)    =", x.var(ddof=0))

---
# Part 4: Getting started on E2 (graded)

`E2.ipynb` has three parts:

| Part | What you do | Check your results with |
| - | - | - |
| 1 – Vectors | sum, mean, median, variance, and a histogram of `petal_length` / `petal_width` | `np.sum`, `np.mean`, `np.median`, `np.var`, `np.histogram` |
| 2 – Matrices | the same statistics on a list of lists, for all values together and per row | the same functions, with `axis=None` / `axis=1` |
| 3 – Scraping | parse the Olympic medalists table into a matrix and compute statistics | spot-check a few rows against the website |

**Important:** implement everything with **plain Python and the `math` module**, then use NumPy only to verify. Also write the *interpretation* cells: full sentences, with units, saying what the number tells you about the flowers or the athletes.

### Clarifications for the assignment text

* The iris measurements are in **centimetres**, not millimetres.
* Task 1.5: the example output `[49, 8, 41, 29, 23], [0.1, 0.58, ...]` is for **petal *width***, not petal length. "precipitation and temperature" means petal length and width.
* Tasks 2.2 and 2.4 say "temperature"; they mean the rows of the iris matrix.
* Task 2.5: the standard deviation is the **square root** of the variance (not the square).
* Use the variance with $\frac{1}{N}$, as in the formula, and compare it with `np.var` (see the warning in Part 3).
* The plotting cell in Task 1.5 draws both histograms into the same figure. Add `plt.figure()` (and `plt.show()`) between the two calls, and replace the placeholder names with your variables.
* Part 3's starter code saves the page with `str(html)`, the same bytes issue we fixed in Part 2. Use `html.decode("utf-8")` if you want the saved file to be readable.

### Hints

**Test with vectors you can compute by hand.** Before using the iris data, try:

| vector | sum | mean | median | variance |
| - | - | - | - | - |
| `[1, 2, 3, 4]` | 10 | 2.5 | 2.5 | 1.25 |
| `[3, 1, 2]` | 6 | 2 | 2 | 0.667 |

Then compare with NumPy. Use `math.isclose(a, b)` instead of `==` for floats, because rounding errors happen (`0.1 + 0.2 == 0.3` is `False`!).

**Histogram (1.5):** work out the width of a bin first. For each value, ask yourself which bin it belongs to, and **what happens with the maximum value** (see the NumPy histogram example in Part 3). Compare your result with `np.histogram(vec, bins=b)`.

**Matrices (Part 2):** your Part 1 functions expect a flat list. What happens if you pass them a list of lists? What do you have to do first? For the per-row versions: what is a row, as far as your Part 1 functions are concerned?

**Scraping (Part 3):**

* "The first visible table" is the first `wikitable` on the page, just like in Part 2.
* **Look at the rows before parsing!** When athletes are tied, the "No." and "Total" cells are **merged across several rows** (`rowspan`). So most rows have **fewer** cells than the header, and the column positions shift. Run the cell below to see it.
* Medal counts arrive as strings: convert them with `int()`.
* Check that all 7 rows of your matrix have the same length at the end.

In [ ]:
# Exploration only (not a solution): how many cells does each row of the Olympic table have?
olympic_cache = Path("olympic-medalists.html")   # different name than in E2, whose starter saves the bytes repr
if not olympic_cache.exists():
    request = urllib.request.Request("https://en.wikipedia.org/wiki/List_of_multiple_Olympic_medalists",
                                     headers={'User-Agent': "firefox"})
    with urllib.request.urlopen(request) as con:
        olympic_cache.write_text(con.read().decode("utf-8"), encoding="utf-8")

olympic_table = BeautifulSoup(olympic_cache.read_text(encoding="utf-8"), "html.parser").find("table", class_="wikitable")
rows = olympic_table.find_all("tr")
print("header:", [th.get_text(strip=True) for th in rows[0].find_all("th")])
print("cells per row:", Counter(len(r.find_all(["th", "td"], recursive=False)) for r in rows[1:]))
for r in rows[3:6]:
    print([c.get_text(strip=True) for c in r.find_all(["th", "td"], recursive=False)])

Now open `E2.ipynb` and get started. Good luck! ?